<a href="https://colab.research.google.com/github/Elwing-Chou/ml0903/blob/main/ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- 筆記語法: markdown語法
- Kaggle網站
- CSV(Comma-Separated Values)
    - 統計學標準表格格式
- 機器學習
    - x(輸入) y(答案)
        - 根據y不一樣, 我們大概會分兩種
            - 分類問題(classification): 答案固定幾種(貓狗預測/畫家預測)
            - 回歸問題(regression):房價預測
- 重點概念
    - 資料的齊全度比你選什麼演算法重要多了
    - 深挖到底

- 演算法
    - tree類型的目前最常用
    - tree型的演算法會自己挑選欄位, 所以你的x越多越好(準確度)

In [4]:
import urllib.request as req

url = "https://github.com/Elwing-Chou/ml0903/raw/refs/heads/main/titanic/train.csv"
req.urlretrieve(url, "train.csv")
url = "https://github.com/Elwing-Chou/ml0903/raw/refs/heads/main/titanic/test.csv"
req.urlretrieve(url, "test.csv")

('test.csv', <http.client.HTTPMessage at 0x7ad750ea9260>)

In [ ]:
import pandas as pd
# pandas表格型態: DataFrame
# 在做統計: 資料分成兩部分
# 訓練資料(train): 給模型學的
# 測試資料(test): 驗證模型調參
# (Optional) 驗證資料: 真實驗證
# 通常我們不會讓模型學到最好(過擬合): 演算法調參數
# 今天的樹類型演算法: 我們會設一個最大深度防止過擬和
# 資料夠不夠:
# ML: 500-1000(還要看問題複雜度決定 x有多少個來決定)
# DL: 10000-100000
data = pd.read_csv("train.csv", encoding="utf-8")
predict = pd.read_csv("test.csv", encoding="utf-8")
predict

In [ ]:
# 1. 準備資料(欄位越多越好)
# 2. 資料預處理(x)
# 2.1 補缺失值
# 2.2 One-hot
# 2.3 想辦法萃取更多欄位
# Family=SibSp+Parch
# 沒做: Fare/Person, Married
# 3. 選擇模型並調參
# 4. 把重要欄位畫圖說故事

In [ ]:
# 把兩個合併再一起處理比較方便
total = pd.concat([data, predict])
# drop: axis=0(row)/1(col)
total = total.drop(["PassengerId", "Survived"], axis=1)
total

In [ ]:
# 我看每個欄位缺多少個值
s = total.isna().sum()
# pandas兩大操作
# 1. 過濾: 把跟資料筆數依樣多True(留下)/False(刪掉)帶入[]資料
s[s > 0]

In [ ]:
# 2. 轉換: def流程
# 少數的留著也沒差, 只是我等等為了保持表格整潔(丟掉)
def flow(s):
    # s = "Braund, Mr. Owen Harris"
    mid = s.split(",")[1].split(".")[0].strip()
    if mid in ["Mr", "Miss", "Mrs", "Master"]:
        return mid
    else:
        return None

name = total["Name"].apply(flow)
# value_counts(): 屬數目
# name.value_counts()
total["Name"] = name
total

In [32]:
record = total["Ticket"].value_counts()
def flow(s):
    return record[s]

total["Ticket"] = total["Ticket"].apply(flow)
total

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,3,Mr,male,22.0,1,0,1,7.2500,NaN,S
1,1,Mrs,female,38.0,1,0,2,71.2833,C85,C
2,3,Miss,female,26.0,0,0,1,7.9250,NaN,S
3,1,Mrs,female,35.0,1,0,2,53.1000,C123,S
4,3,Mr,male,35.0,0,0,1,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...
413,3,Mr,male,NaN,0,0,1,8.0500,NaN,S
414,1,None,female,39.0,0,0,3,108.9000,C105,C
415,3,Mr,male,38.5,0,0,1,7.2500,NaN,S
416,3,Mr,male,NaN,0,0,1,8.0500,NaN,S


In [46]:
# Cabin處理
def flow(s):
    if pd.isna(s):
        return s
    else:
        return s[0]
total["Cabin"] = total["Cabin"].apply(flow)

In [44]:
# 補缺失值: 補最可能的
# 把x分成兩大類
# 1. 類別型: Pclass, Name, Sex, Cabin, Embarked
# 類別: 最常出現的值(我通常選擇不補)
# 2. 數值型(大小): Age, SibSp, Parch, Ticket(#), Fare
# 數值: 中位數
# Pclass特別: 1 1 2 2 3 3 3
# 補2(2.5), 但其實該是3
# 拿出所有數字類型的欄位
num_cols = total.dtypes != "object"
num_cols = total.dtypes[num_cols].drop("Pclass")
# 取出數值類型的那些欄位
med = total[num_cols.index].median()
# 真的補回去
total = total.fillna(med)
# 再看一次到底缺多少個
s = total.isna().sum()
s[s > 0]

,0
Name,34
Cabin,1014
Embarked,2


In [47]:
# 初步處理(把欄位該處理的處理一下)->數值中位數填補結束
total

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,3,Mr,male,22.0,1,0,1,7.2500,NaN,S
1,1,Mrs,female,38.0,1,0,2,71.2833,C,C
2,3,Miss,female,26.0,0,0,1,7.9250,NaN,S
3,1,Mrs,female,35.0,1,0,2,53.1000,C,S
4,3,Mr,male,35.0,0,0,1,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...
413,3,Mr,male,28.0,0,0,1,8.0500,NaN,S
414,1,None,female,39.0,0,0,3,108.9000,C,C
415,3,Mr,male,38.5,0,0,1,7.2500,NaN,S
416,3,Mr,male,28.0,0,0,1,8.0500,NaN,S
